[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abigailhaddad/fedscope_new/blob/main/demo.ipynb)

# EHRI Federal Workforce Data Explorer

This notebook loads federal workforce data from HuggingFace and lets you explore trends over time:
- **Accessions** - New federal hires
- **Separations** - Federal employee departures
- **Employment** - Point-in-time workforce snapshots

**No authentication required** - all datasets are public. Available months are discovered automatically.

In [ ]:
!pip install -q duckdb pandas plotly huggingface_hub great_tables

In [ ]:
import re
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from huggingface_hub import list_repo_files
from great_tables import GT
import base64
from IPython.display import HTML, display

def download_csv(df, filename):
    csv = df.to_csv(index=False)
    b64 = base64.b64encode(csv.encode()).decode()
    display(HTML(f'<a href="data:file/csv;base64,{b64}" download="{filename}" style="background:#4CAF50;color:white;padding:8px 16px;text-decoration:none;border-radius:4px;display:inline-block;margin:10px 0;">Download {filename}</a>'))

## 1. Discover Available Files

We query the HuggingFace repo to find all available months automatically.

In [ ]:
HF_REPO = "abigailhaddad/opm-federal-workforce"
BASE_URL = f"https://huggingface.co/datasets/{HF_REPO}/resolve/main"

all_files = list(list_repo_files(HF_REPO, repo_type="dataset"))
parquet_files = [f for f in all_files if f.endswith(".parquet")]

def get_urls(data_type):
    """Get sorted list of HF URLs for a given data type.

    Handles versioned filenames (accessions_202511_v3.parquet).
    For each month, uses only the highest version available.
    """
    pattern = re.compile(rf"^{data_type}/{data_type}_(\d{{6}})_v(\d+)\.parquet$")
    # Also accept legacy unversioned files
    legacy = re.compile(rf"^{data_type}/{data_type}_(\d{{6}})\.parquet$")

    # {yyyymm: (version, path)}
    best = {}
    for f in parquet_files:
        m = pattern.match(f)
        if m:
            yyyymm, ver = m.group(1), int(m.group(2))
            if yyyymm not in best or ver > best[yyyymm][0]:
                best[yyyymm] = (ver, f)
            continue
        m = legacy.match(f)
        if m:
            yyyymm = m.group(1)
            if yyyymm not in best:
                best[yyyymm] = (0, f)

    months = sorted(best.keys())
    urls = [f"{BASE_URL}/{best[m][1]}" for m in months]
    return urls, months

acc_urls, acc_months = get_urls("accessions")
sep_urls, sep_months = get_urls("separations")
emp_urls, emp_months = get_urls("employment")

print(f"Accessions:  {len(acc_urls)} files  ({acc_months[0]} - {acc_months[-1]})")
print(f"Separations: {len(sep_urls)} files  ({sep_months[0]} - {sep_months[-1]})")
print(f"Employment:  {len(emp_urls)} files  ({emp_months[0]} - {emp_months[-1]})")

## 2. Load Accessions & Separations

These are small (~0.2 MB/month parquet) so we load all available months into DuckDB. Queries after this are instant.

In [ ]:
%%time
N_MONTHS = 2  # most recent months per data type (6 files total)
db = duckdb.connect()

acc_list = ", ".join(f"'{u}'" for u in acc_urls[-N_MONTHS:])
db.execute(f"CREATE TABLE accessions AS SELECT * FROM read_parquet([{acc_list}])")

sep_list = ", ".join(f"'{u}'" for u in sep_urls[-N_MONTHS:])
db.execute(f"CREATE TABLE separations AS SELECT * FROM read_parquet([{sep_list}])")

emp_list = ", ".join(f"'{u}'" for u in emp_urls[-N_MONTHS:])
db.execute(f"CREATE VIEW employment AS SELECT * FROM read_parquet([{emp_list}])")

acc_count = db.execute("SELECT SUM(CAST(count AS INTEGER)) FROM accessions").fetchone()[0]
sep_count = db.execute("SELECT SUM(CAST(count AS INTEGER)) FROM separations").fetchone()[0]
emp_count = db.execute("SELECT SUM(CAST(count AS INTEGER)) FROM employment").fetchone()[0]
print(f"Loaded {N_MONTHS} months: {acc_months[-N_MONTHS]} - {acc_months[-1]}")
print(f"  Accessions: {acc_count:,} records")
print(f"  Separations: {sep_count:,} records")
print(f"  Employment: {emp_count:,} records")

## 3. Monthly Overview

In [ ]:

# Summary table: accessions & separations totals per month
acc_totals = db.execute("""
    SELECT personnel_action_effective_date_yyyymm as month,
           SUM(CAST(count AS INTEGER)) as accessions
    FROM accessions GROUP BY month ORDER BY month
""").df()

sep_totals = db.execute("""
    SELECT personnel_action_effective_date_yyyymm as month,
           SUM(CAST(count AS INTEGER)) as separations
    FROM separations GROUP BY month ORDER BY month
""").df()

summary = acc_totals.merge(sep_totals, on='month')
summary['net'] = summary['accessions'] - summary['separations']
summary['Month'] = pd.to_datetime(summary['month'], format='%Y%m').dt.strftime('%B %Y')

display(
    GT(summary[['Month', 'accessions', 'separations', 'net']]
       .rename(columns={'accessions': 'Accessions', 'separations': 'Separations', 'net': 'Net Change'}))
    .tab_header(title="Federal Workforce: Monthly Overview")
    .fmt_integer(columns=['Accessions', 'Separations', 'Net Change'])
)
download_csv(summary[['Month', 'accessions', 'separations', 'net']], 'monthly_overview.csv')

In [ ]:

# Accessions: 2-month comparison by agency
acc_agency = db.execute("""
    SELECT agency, personnel_action_effective_date_yyyymm as month,
           SUM(CAST(count AS INTEGER)) as count
    FROM accessions GROUP BY agency, month
""").df()

months = sorted(acc_agency['month'].unique())
prev_m, curr_m = months[-2], months[-1]
prev_label = pd.to_datetime(prev_m, format='%Y%m').strftime('%B %Y')
curr_label = pd.to_datetime(curr_m, format='%Y%m').strftime('%B %Y')

pivot = acc_agency.pivot(index='agency', columns='month', values='count').fillna(0).reset_index()
pivot.columns = ['Agency', prev_label, curr_label]
pivot['Change'] = (pivot[curr_label] - pivot[prev_label]).astype(int)
pivot['% Change'] = (pivot['Change'] / pivot[prev_label].replace(0, float('nan')) * 100).round(1)

top20 = pivot.reindex(pivot['Change'].abs().nlargest(20).index).reset_index(drop=True)

display(
    GT(top20)
    .tab_header(title=f"Accessions by Agency: {prev_label} vs {curr_label}",
                subtitle="Top 20 by absolute change")
    .fmt_integer(columns=['Agency', prev_label, curr_label, 'Change'])
    .fmt_number(columns=['% Change'], decimals=1)
)
download_csv(pivot.sort_values('Change', key=abs, ascending=False), 'accessions_comparison.csv')

## 4. Accessions by Agency — Heatmap

In [ ]:

# Heatmap: top 15 agencies by accessions across available months
month_labels = {m: pd.to_datetime(m, format='%Y%m').strftime('%b %Y')
                for m in acc_agency['month'].unique()}
top15 = acc_agency.groupby('agency')['count'].sum().nlargest(15).index.tolist()

heat = (acc_agency[acc_agency['agency'].isin(top15)]
        .assign(month_label=lambda d: d['month'].map(month_labels))
        .pivot(index='agency', columns='month_label', values='count')
        .fillna(0))
# Sort by most recent month descending
heat = heat.sort_values(heat.columns[-1], ascending=False)

fig = px.imshow(
    heat, text_auto=',d', color_continuous_scale='Blues',
    title=f'Accessions — Top 15 Agencies',
    labels=dict(x='Month', y='Agency', color='Hires'),
)
fig.update_layout(height=520, xaxis_title='', yaxis_title='',
                  coloraxis_showscale=False)
fig.show()

## 5. Columns & Available Values

In [ ]:
print("Accessions columns:", db.execute("DESCRIBE accessions").df()['column_name'].tolist())
print("\nSeparations columns:", db.execute("DESCRIBE separations").df()['column_name'].tolist())

In [ ]:
for field in ['agency', 'occupational_group', 'education_level', 'duty_station_state']:
    result = db.execute(f"""
        SELECT {field} as value, SUM(CAST(count AS INTEGER)) as n
        FROM accessions WHERE {field} IS NOT NULL AND {field} != ''
        GROUP BY {field} ORDER BY n DESC LIMIT 8
    """).df()
    print(f"\n{field}:")
    for _, row in result.iterrows():
        print(f"  {row['value']}: {row['n']:,}")

In [ ]:
%%time
N_MONTHS = 2
recent_emp_urls = emp_urls[-N_MONTHS:]
recent_emp_months = emp_months[-N_MONTHS:]
print(f"Loading employment: {recent_emp_months[0]} - {recent_emp_months[-1]}")

emp_list = ", ".join(f"'{u}'" for u in recent_emp_urls)
db.execute(f"CREATE OR REPLACE VIEW employment AS SELECT * FROM read_parquet([{emp_list}])")

emp_count = db.execute("SELECT SUM(CAST(count AS INTEGER)) FROM employment").fetchone()[0]
print(f"Loaded {emp_count:,} employee records")
print("Columns:", db.execute("DESCRIBE employment").df()['column_name'].tolist())

In [ ]:

# Employment snapshot: top 20 agencies in the most recent month
latest_month = emp_months[-1]
latest_label = pd.to_datetime(latest_month, format='%Y%m').strftime('%B %Y')

top_emp = db.execute(f"""
    SELECT agency, SUM(CAST(count AS INTEGER)) as employees
    FROM employment WHERE snapshot_yyyymm = '{latest_month}'
    GROUP BY agency ORDER BY employees DESC LIMIT 20
""").df()

fig = px.bar(
    top_emp, x='employees', y='agency', orientation='h',
    title=f'Top 20 Agencies by Employment — {latest_label}',
    color='employees', color_continuous_scale='Blues',
)
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'},
                  xaxis=dict(tickformat=','), showlegend=False, coloraxis_showscale=False)
fig.show()
download_csv(top_emp, f'employment_{latest_month}.csv')

## 7. Try Your Own Queries

Three DuckDB tables are loaded: `accessions`, `separations`, `employment`.

**Common fields:**
- `agency` — e.g. `DEPARTMENT OF DEFENSE`
- `duty_station_state` — e.g. `CALIFORNIA`
- `occupational_group` — e.g. `INFORMATION TECHNOLOGY GROUP`
- `occupational_series` — e.g. `2210`
- `education_level`, `age_bracket`, `supervisory_status`
- `personnel_action_effective_date_yyyymm` (accessions/separations)
- `snapshot_yyyymm` (employment)
- `count` — number of people in that combination of values

All counts are stored as strings — cast with `CAST(count AS INTEGER)`.

In [ ]:
# Example: Army separations by month
df = db.execute(f"""
    SELECT personnel_action_effective_date_yyyymm as month,
           SUM(CAST(count AS INTEGER)) as departures
    FROM separations
    WHERE agency = 'DEPARTMENT OF THE ARMY'
      AND personnel_action_effective_date_yyyymm IN ({sep_months_filter})
    GROUP BY month ORDER BY month
""").df()
df['date'] = pd.to_datetime(df['month'], format='%Y%m')

fig = go.Figure(go.Scatter(x=df['date'], y=df['departures'], mode='lines+markers',
    line=dict(color='#e74c3c')))
fig.update_layout(title='Army Separations by Month', template='plotly_white',
    yaxis=dict(tickformat=','), height=400)
fig.show()